在qust中，一个策略其实就是一个算子。算子其实就是一个函数，输入一个DataFrame，返回一个DataFrame

这样有几个好处

1. 利用现有的算子用搭积木的方式组成一个策略

2. 策略逻辑与数据完全分离，方便把同一个策略放在不同的数据验证
 

比如我现在写了一个策略:


In [1]:
import sys
sys.path.append("/root/otters/otters-py/python")
import polars as pl
import qust as qs
from qust import col
import datetime as dt
import numpy as np
from importlib import reload as rl

In [4]:
class MyStra(qs.UdfRow):

    def __init__(self):
        self.sum = 0.0
        self.count = 0.0

    def output_schema(self, input_schema):
        return [("mean_res", pl.Float64)]
    
    def update(self, value):
        self.sum += value
        self.count += 1.0

    def calc(self):
        return [self.sum / self.count]

my_stra = col("a").mean().rolling(3).udf.row(MyStra).expanding()

这个`my_stra`我在期货上测试有效，但是我目前没有股票数据或者没有精力测股票

那我可以先把这个`my_stra`存储到本地：

In [5]:
my_stra.save("my_stra")

然后我把这个文件给其他人，他可以这样读取策略逻辑，然后在其他数据上测试：

In [6]:
qs.Expr.load("my_stra")

Expr { inner: ExprExpanding { accumulator: Expr { inner: ExprPip { child: Expr { inner: ExprRolling { window_size: 3, min_samples: 3, accumulator: Expr { inner: ExprPip { child: Expr { inner: ExprCutBwCtx { e: Expr { inner: ExprPlExpr { inner: col("a") } } } }, parent: Expr { inner: ExprMean } } } } }, parent: Expr { inner: ExprUdfRow { inner: PyAnyArc { inner: Py(0x556cc19c82d0) } } } } } } }

如果我想给其他人测试但是又不想暴露策略，我可以这样存储

In [9]:
my_stra_private = my_stra.private()
my_stra_private

Expr { inner: 加密算子，不可见 }

In [10]:
my_stra_private.save("my_stra_private")

In [11]:
qs.Expr.load("my_stra_private")

Expr { inner: 加密算子，不可见 }

这种模式很有用，比如对我个人来说，目前有一些信号类策略开仓信号都不错，但是平仓信号比较差，如果能交易到平仓信号，双赢

当然这种模式也不仅限与策略或者因子之类，还有一些组合优化、仓位管理的算子也很适用这种模式